## Data File

Downloaded the roman empire version used in the Yandex Research from https://www.kaggle.com/datasets/nbroad/wiki-20220301-en/data
Found the article from 26 parquet file by scanning them and extracting the exact document. 

In [6]:
#with open("../data/roman_empire_wikipedia_2022.txt", "r", encoding="utf-8") as f:
with open("../data/roman_empire_wikipedia_20220301.txt", "r", encoding="utf-8") as f:
    roman_empire = f.read()

## Roman - Empire Graph 

https://arxiv.org/pdf/2302.11640

This dataset is based on the Roman Empire article from English Wikipedia, which was selected since it is one of the longest articles on Wikipedia. The text was retrieved from the English Wikipedia 2022.03.01 dump from Lhoest et al. (2021). Each node in the graph corresponds to one (non-unique) word in the text. Thus, the number of nodes in the graph is equal to the article’s length. Two words are connected with an edge if at least one of the following two conditions holds:
either these words follow each other in the text, or these words are connected in the dependency tree of the sentence (one word depends on the other). Thus, the graph is a chain graph with additional shortcut edges corresponding to syntactic dependencies between words. 

In [8]:
import spacy
import networkx as nx
import torch
import numpy as np
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaModel

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device)
model.eval()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1287.46it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [9]:
nlp = spacy.load("en_core_web_md")
doc = nlp(roman_empire)

# Use a Graph (undirected) to ensure A-B is only one edge
G = nx.Graph() 

#Not punctuation and space, only words
valid_tokens = [t for t in doc if not t.is_punct and not t.is_space]
token_lookup = {t.i: j for j, t in enumerate(valid_tokens)}

#For Roberta
valid_token_ids = [t.i for t in valid_tokens]
node_features = np.zeros((len(valid_tokens), 768), dtype=np.float32)
node_id_map = {t.i: idx for idx, t in enumerate(valid_tokens)}


for j, token in enumerate(valid_tokens):
    G.add_node(j)
    
    # Sequential connection (The Chain)
    if j < len(valid_tokens) - 1:
        G.add_edge(j, j + 1)
    
    #Syntactic Dependency (The Shortcuts)
    if token.head.i in token_lookup:
        target_j = token_lookup[token.head.i]
        if target_j != j:
            # .add_edge in nx.Graph() automatically handles "at least one"
            # If the edge already exists from the chain logic, it won't add a duplicate
            G.add_edge(j, target_j)

print(f"Graph: {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")




Graph: 22600 nodes and 32684 edges.


# Encode the Node Feature

In [10]:
for sent in doc.sents:
    # Tokenize the whole sentence
    inputs = tokenizer(sent.text, return_tensors="pt", truncation=True, padding=True).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Shape: [1, num_tokens, 768]
    embeddings = outputs.last_hidden_state.squeeze(0)

    word_ids = inputs.word_ids() 

    for word_idx in set(w for w in word_ids if w is not None):
        # Find which tokens belong to this word
        token_indices = [i for i, w in enumerate(word_ids) if w == word_idx]
        
        # Average the subword vectors
        avg_vec = embeddings[token_indices].mean(dim=0).cpu().numpy()
        
        # We need to map this word back to our specific spaCy token
        # This is a simplified mapping; in a production script, you'd align 
        # the spaCy token text with the RoBERTa word index.
        # For now, we'll iterate through the sentence's tokens:
        if word_idx < len(sent):
            spacy_token = sent[word_idx]
            if spacy_token.i in node_id_map:
                node_idx = node_id_map[spacy_token.i]
                node_features[node_idx] = avg_vec
    
embeddings_df = pd.DataFrame(node_features)
embeddings_df.to_csv('../data/node_features_roberta.csv', index=False, header=False)
print(f"Done! Created feature matrix: {node_features.shape}")

Done! Created feature matrix: (22600, 768)


## Identify the Node Class

In [11]:
from collections import Counter

# Get all raw syntactic roles from your valid_tokens list
raw_roles = [token.dep_ for token in valid_tokens]

role_counts = Counter(raw_roles)

# Identify the 17 most frequent roles
top_17_roles = [role for role, count in role_counts.most_common(17)]

for i, role in enumerate(top_17_roles):
    print(f"Class {i}: {role} ({role_counts[role]} occurrences)")

Class 0: pobj (3162 occurrences)
Class 1: prep (3137 occurrences)
Class 2: det (2510 occurrences)
Class 3: amod (2376 occurrences)
Class 4: conj (1383 occurrences)
Class 5: nsubj (1224 occurrences)
Class 6: cc (1080 occurrences)
Class 7: compound (928 occurrences)
Class 8: ROOT (923 occurrences)
Class 9: dobj (872 occurrences)
Class 10: advmod (795 occurrences)
Class 11: aux (446 occurrences)
Class 12: auxpass (370 occurrences)
Class 13: appos (362 occurrences)
Class 14: nsubjpass (329 occurrences)
Class 15: poss (321 occurrences)
Class 16: relcl (296 occurrences)


In [12]:
role_to_class = {role: i for i, role in enumerate(top_17_roles)}

node_labels = []
for role in raw_roles:
    # Get class ID from mapping, default to 17 (18th class) if not in top 17
    class_id = role_to_class.get(role, 17)
    node_labels.append(class_id)

import numpy as np
y = np.array(node_labels)

print(f"\nNode labels shape: {y.shape}")
print(f"Unique classes found: {np.unique(y)}")


Node labels shape: (22600,)
Unique classes found: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]


# Generate the Edge Index

In [13]:
edges = list(G.edges())
edge_list_df = pd.DataFrame(edges, columns=['source', 'target'])
edge_list_df = edge_list_df.astype(int)
edge_list_df.to_csv('../data/edge_list.txt', sep=' ', index=False, header=False)


# Create Labels

In [14]:
import pandas as pd
import numpy as np

labels_df = pd.Series(y)

# Save to labels.csv
labels_df.to_csv('../data/labels.csv', index=False, header=False)


# Generate NPZ

In [17]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from scripts.create_npz import create_npx

create_npx.create_npz_dataset(node_features_file="../data/node_features_roberta.csv", labels_file='../data/labels.csv', edges_file='../data/edge_list.txt', num_splits=10, output_file_name='../data/roman-roberta.npz')


Verified Split 0 - Train: 13560, Val: 4520, Test: 4520
no of nodes 22600
no of features per node 768
no of node labels, should matach no of nodes 22600
no of edges 32684
